# Expert-action imitation pretrain on Colab

Trains a small `ActionDecoder` (3 linear heads: source planet, target planet, expert_acted) on top of the cross-entity attention. Mirrors `train_cross_entity_colab.ipynb` — pulls the same `code.tgz` / `data.tgz` / `weights.tgz` from `gs://orbit-wars-shipping/`, runs Stage 0 (frozen) then Stages 1–4 (gradual unfreeze), then pushes the run dir back to GCS.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. The Google account you sign into Colab with must own (or have read/write on) the GCP project `analog-receiver-489214-e9` and bucket `gs://orbit-wars-shipping/`.
3. `weights.tgz` must include a `cross_entity_best.pt` — the action decoder is seeded from its cross-attention sub-module. If `pack_for_gpu.sh` was run with `INCLUDE_CROSS_ENTITY=1`, that's already done.

Total runtime: ~3 min setup + ~10 min frozen + ~30 min gradual = ~45 min.

## 1. Verify GPU

Free Colab sometimes hands out CPU runtimes silently. Abort early if so — no point training when we have a CPU at home.

In [ ]:
import torch, sys
if not torch.cuda.is_available():
    sys.exit('No GPU runtime — Runtime → Change runtime type → T4 GPU, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Authenticate to GCP & pull tarballs

`google.colab.auth.authenticate_user()` makes the credentials of your Google account available to `gsutil` — no service-account keys to manage.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET = 'gs://orbit-wars-shipping'

!gcloud config set project {PROJECT}

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

for name in ('code.tgz', 'data.tgz', 'weights.tgz'):
    !gsutil cp {BUCKET}/{name} .

## 3. Unpack + install

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz

# Sanity: cross_entity_best.pt must be present so the action decoder can
# seed its cross-attention from a trained cross-entity checkpoint.
import glob
ce = glob.glob('data/runs/cross_entity/*/cross_entity_best.pt')
if not ce:
    raise SystemExit(
        'no cross_entity_best.pt found — repack with '
        '`INCLUDE_CROSS_ENTITY=1 ./scripts/pack_for_gpu.sh` and reupload.'
    )
print('cross_entity ckpt(s):'); [print(' ', p) for p in ce]
!ls -la

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

## 4. Sanity-check imports + dataset coverage

Catches version-skew before burning GPU time. Also reports how many episodes have all 5 CSVs (planet+fleet+entity+cross_entity+action) — only those train.

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.expert_action import (
    ActionDecoder, ActionSnapshotDataset, ActionTrainStack,
    train_frozen, train_gradual_unfreeze,
)
from agents.transformer_v1.paths import (
    FLEET_RUNS_DIR, PLANET_RUNS_DIR, ENTITY_RUNS_DIR,
    CROSS_ENTITY_RUNS_DIR, ACTION_RUNS_DIR,
    FLEET_DATASET_DIR, PLANET_DATASET_DIR, ENTITY_DATASET_DIR,
    CROSS_ENTITY_DATASET_DIR, ACTION_DATASET_DIR,
)
from pathlib import Path

for d in (FLEET_RUNS_DIR, PLANET_RUNS_DIR, ENTITY_RUNS_DIR,
          CROSS_ENTITY_RUNS_DIR, ACTION_RUNS_DIR):
    runs = sorted(p for p in Path(d).iterdir() if p.is_dir()) if Path(d).exists() else []
    print(f'{d}:')
    for r in runs:
        print(f'  {r.name}')

# Coverage: stems with all 5 CSVs.
stems = {p.stem.removeprefix('action_') for p in ACTION_DATASET_DIR.glob('action_*.csv')}
for d, prefix in (
    (PLANET_DATASET_DIR, 'planet_'), (FLEET_DATASET_DIR, 'fleet_'),
    (ENTITY_DATASET_DIR, 'entity_'), (CROSS_ENTITY_DATASET_DIR, 'cross_entity_'),
):
    stems &= {p.stem.removeprefix(prefix) for p in d.glob(f'{prefix}*.csv')}
print(f'\nepisodes with all 5 CSVs: {len(stems)}')
print('imports OK')

## 5. Stage 0 — train decoder on frozen encoders

The action decoder (3 linear heads) is brand-new; train it for a few epochs with everything else frozen so it starts from a sensible point before we thaw lower layers in Stage 1+. Outputs land in `data/runs/action/<timestamp>/action_best.pt`.

In [ ]:
%cd {WORK}
!python -m agents.transformer_v1.pretrain.expert_action \
    --train-mode frozen \
    --epochs 10 \
    --batch-size 64 --num-workers 2 \
    --eval-every 1 --device cuda

## 6. Stages 1–4 — gradual unfreeze

Resumes from the latest `action_best.pt` (the Stage 0 output), then progressively thaws cross-attention → entity encoder → planet/fleet top-halves → full stack. `--stage-epochs 5,5,5,5` runs all four stages; pass fewer entries to skip later stages.

In [ ]:
%cd {WORK}
!python -m agents.transformer_v1.pretrain.expert_action \
    --train-mode gradual-unfreeze \
    --stage-epochs 5,5,5,5 \
    --batch-size 64 --num-workers 2 \
    --eval-every 1 --device cuda

## 7. Push results back to GCS

Tars all action run dirs (Stage 0 + gradual) and uploads back to the same bucket so we can `gsutil cp` from local.

In [ ]:
import time
from pathlib import Path

action_root = Path(WORK, 'data', 'runs', 'action')
runs = sorted(p for p in action_root.iterdir() if p.is_dir())
if not runs:
    raise SystemExit('no run dir found under data/runs/action/')
print('pushing', len(runs), 'run dir(s):')
for r in runs:
    print(' ', r.relative_to(WORK))

stamp = time.strftime('%Y%m%d-%H%M%S')
tar_name = f'action_runs_{stamp}.tgz'
%cd {action_root}
!tar czf /tmp/{tar_name} {' '.join(r.name for r in runs)}
!gsutil cp /tmp/{tar_name} {BUCKET}/{tar_name}
print(f'\nPull from local with:\n  gsutil cp {BUCKET}/{tar_name} .')

## 8. Quick test-set summary

Print the per-head test metrics (loss, top-1 acc, top-3 acc) from the most recent run. Top-3 accuracy is the practical metric for source/target — getting the right planet within the top-3 candidates is what matters for downstream agent integration.

In [ ]:
import json
latest = runs[-1]
summary = json.loads((latest / 'test_summary.json').read_text())
print(f'{latest.name}\n')
print(f'{"head":<24s}  {"loss":>8s}  {"acc":>6s}  {"top3":>6s}')
for name, m in summary.items():
    acc = f'{m["acc"]:.3f}' if 'acc' in m else '-'
    top3 = f'{m["acc_top3"]:.3f}' if 'acc_top3' in m else '-'
    print(f'{name:<24s}  {m["loss"]:>8.4f}  {acc:>6s}  {top3:>6s}')